In [ ]:
#ISENCAO

In [154]:
from pyspark.sql.functions import lit
from pyspark.sql import functions as f
from pyspark.sql.functions import *
import pandas as pd
import logging

In [155]:
bi_isencao_matricula_nao_paga = spark.read.parquet('s3a://cdl-planejamentocaixa-us-east-1-660894085742-dev-integration/BI_ISENCAO_MATRICULA_NAO_PAGA/')

In [156]:
bi_isencao_credito_maior_debito = spark.read.parquet('s3a://cdl-planejamentocaixa-us-east-1-660894085742-dev-integration/BI_ISENCAO_CREDITO_MAIOR_DEBITO/')

In [157]:
bi_carteira_cobranca = spark.read.parquet('s3a://cdl-planejamentocaixa-us-east-1-660894085742-dev-integration/BI_CARTEIRA_COBRANCA/')

In [158]:
bi_isencao = spark.read.parquet('s3a://cdl-planejamentocaixa-us-east-1-660894085742-dev-integration/ISENCAO_RAID/')

In [159]:
def join_isencao(bi_carteira_cobranca, isencao_raid):
    df_isencao = (bi_carteira_cobranca.join(isencao_raid,
                                  bi_carteira_cobranca['COD_CONTRATO'] == isencao_raid['COD_CONTRATO_ISENCAO'], "inner"))    
    df_isencao = df_isencao.select(f.col('COD_CONTRATO_ISENCAO'),f.col('DSC_MOTIVO_ISENCAO'))
    
    return df_isencao

In [160]:
def join_isencao_matricula_nao_paga(bi_carteira_cobranca, bi_isencao_matricula_nao_paga):
    df_isencao_matricula = (bi_carteira_cobranca.join(bi_isencao_matricula_nao_paga,
                                  bi_carteira_cobranca['COD_CPFCGC_PES'] == bi_isencao_matricula_nao_paga['COD_RA_IMNP'], "inner"))
    df_isencao_matricula = df_isencao_matricula.select(f.col('COD_CONTRATO').alias('COD_CONTRATO_MAT'),f.col('DSC_MOTIVO_MNP').alias('DSC_MOTIVO_MAT'))
    
    return df_isencao_matricula

In [161]:
def join_isencao_cmd(bi_carteira_cobranca, bi_isencao_credito_maior_debito):
    df_isencao_cmd = (bi_carteira_cobranca.join(bi_isencao_credito_maior_debito,
                                  bi_carteira_cobranca['COD_CPFCGC_PES'] == bi_isencao_credito_maior_debito['COD_RA_ICMD'], "inner"))
    df_isencao_cmd = df_isencao_cmd.select(f.col('COD_CONTRATO').alias('COD_CONTRATO_CRE'),f.col('STA_COBRANCA_CMD').alias('DSC_MOTIVO_CRE'))
    
    return df_isencao_cmd

In [162]:
df_isencao_cmd = join_isencao_cmd(bi_carteira_cobranca, bi_isencao_credito_maior_debito)
df_isencao_matricula = join_isencao_matricula_nao_paga(bi_carteira_cobranca, bi_isencao_matricula_nao_paga)
df_isencao = join_isencao(bi_carteira_cobranca, bi_isencao)
df_cd_carteira = bi_carteira_cobranca

In [163]:
df_isencao_matricula.printSchema()

root
 |-- COD_CONTRATO_MAT: string (nullable = true)
 |-- DSC_MOTIVO_MAT: string (nullable = true)



In [164]:
df_isencao.printSchema()

root
 |-- COD_CONTRATO_ISENCAO: string (nullable = true)
 |-- DSC_MOTIVO_ISENCAO: string (nullable = true)



In [165]:
df_cd_carteira = (df_cd_carteira.join(df_isencao,
                                  df_cd_carteira['COD_CONTRATO'] == df_isencao['COD_CONTRATO_ISENCAO'], "left"))

In [166]:
df_cd_carteira = (df_cd_carteira.join(df_isencao_matricula,
                                  df_cd_carteira['COD_CONTRATO'] == df_isencao_matricula['COD_CONTRATO_MAT'], "left"))

In [167]:
    df_cd_carteira = (df_cd_carteira.join(df_isencao_cmd,
                                      df_cd_carteira['COD_CONTRATO'] == df_isencao_cmd['COD_CONTRATO_CRE'], "left"))

In [171]:
df_cd_carteira = (
    df_cd_carteira
    .withColumn("DSC_SITUACAO_COBRANCA", 
        f.when(f.col("DSC_MOTIVO_MAT").isNotNull(), 'MATRICULA NAO PAGA')
        .when((f.col("DSC_MOTIVO_ISENCAO").isNotNull()),f.col("DSC_MOTIVO_ISENCAO"))
        .when((f.col("CLS_SITUACAO_CONTRATO_ALUNO").isin('12','14','17','20','22','23','33','36','37','Err')),"FORA COBRANCA SITUACAO ACADEMICA")
        .when((f.col("DSC_MOTIVO_CRE").isNotNull()), 'CREDITO MAIOR QUE DEBITO')
        .otherwise(f.lit('EM COBRANCA'))
))

In [172]:
df_cd_carteira.select(f.col('DSC_SITUACAO_COBRANCA')).distinct().show()

+---------------------+
|DSC_SITUACAO_COBRANCA|
+---------------------+
| TRAVA SISTEMICA C...|
| TRAVA SISTEMICA C...|
| TRAVA SISTEMICA C...|
| TRAVA SISTEMICA C...|
| TRAVA SISTEMICA C...|
| TRAVA SISTEMICA C...|
| TRAVA SISTEMICA C...|
| TRAVA SISTEMICA C...|
| TRAVA SISTEMICA C...|
| TRAVA SISTEMICA C...|
| TRAVA SISTEMICA C...|
| TRAVA SISTEMICA C...|
| TRAVA SISTEMICA C...|
| TRAVA SISTEMICA C...|
| TRAVA SISTEMICA C...|
| TRAVA SISTEMICA C...|
| TRAVA SISTEMICA C...|
| TRAVA SISTEMICA C...|
| TRAVA SISTEMICA C...|
| TRAVA SISTEMICA C...|
+---------------------+
only showing top 20 rows



In [147]:
credito = (df_cd_carteira.filter(f.col('DSC_SITUACAO_COBRANCA') == 'CREDITO MAIOR QUE DEBITO'))

In [149]:
matricula = (df_cd_carteira.filter(f.col('DSC_SITUACAO_COBRANCA') == "MATRICULA NAO PAGA"))

In [173]:
isencao = (df_cd_carteira.filter(f.col('DSC_SITUACAO_COBRANCA') == "FORA COBRANCA SITUACAO ACADEMICA"))

In [174]:
isencao.count()

480619

In [151]:
df_cd_carteira.select(f.col('DSC_SITUACAO_COBRANCA')).distinct().show()

+---------------------+
|DSC_SITUACAO_COBRANCA|
+---------------------+
|          EM COBRANCA|
| FORA DE COBRANCA ...|
+---------------------+



In [153]:
df_isencao.count()

143631

In [142]:
df_cd_carteira.select(f.col('DSC_SITUACAO_COBRANCA')).show()

+---------------------+
|DSC_SITUACAO_COBRANCA|
+---------------------+
|                 null|
|                 null|
|                 null|
|                 null|
|                 null|
|                 null|
|                 null|
|                 null|
|                 null|
|                 null|
|                 null|
|                 null|
|                 null|
|                 null|
|                 null|
| FORA DE COBRANCA ...|
|                 null|
|                 null|
|                 null|
| FORA DE COBRANCA ...|
+---------------------+
only showing top 20 rows



In [ ]:
df_cd_carteira = df_cd_carteira.select(f.col("*"), f.expr("CASE WHEN COD_STCO_ACDM in ('C','P') THEN 'ATIVO' " + 
                                                          "WHEN COD_STCO_ACDM in ('D') THEN 'INATIVO'" +
                                                          "ELSE 'NAO ENCONTRADO' END").alias("STA_STCO_ACDM"))

In [ ]:
#nova coluna - ISENCAO
df_cd_carteira =  (df_cd_carteira.withColumn("DSC_SITUACAO_COBRANCA", f.when(f.col("DSC_SITUACAO_COBRANCA").isNull(),
                                                     f.lit("EM COBRANCA"))
                                            .otherwise(
                                                f.col('DSC_SITUACAO_COBRANCA')
                                            )))

In [83]:
df_cd_carteira.printSchema()

root
 |-- NOM_AGENCIA_EXTERNA: string (nullable = true)
 |-- DAT_PROCESSAMENTO: date (nullable = true)
 |-- CLS_MARCA_CONTRATO: string (nullable = true)
 |-- COD_FILIAL: string (nullable = true)
 |-- DSC_FILIAL: string (nullable = true)
 |-- CLS_SITUACAO_CONTRATO_ALUNO: string (nullable = true)
 |-- DSC_SITUACAO_CONTRATO: string (nullable = true)
 |-- COD_CLIENTE: string (nullable = true)
 |-- COD_CONTRATO: string (nullable = true)
 |-- COD_CONTRATO_ORIGEM: string (nullable = true)
 |-- COD_CPFCGC_PES: string (nullable = true)
 |-- NOM_CLIENTE: string (nullable = true)
 |-- CLS_SCORE_COLLECTION: string (nullable = true)
 |-- CLS_CLASSE_DEBITO: string (nullable = true)
 |-- CLS_CLUSTER: string (nullable = true)
 |-- CLS_SCORE_GLOBAL: string (nullable = true)
 |-- CLS_SCORE_ENGAJAMENTO: string (nullable = true)
 |-- DAT_INICIO_CURSO: string (nullable = true)
 |-- DAT_FIM_CURSO: string (nullable = true)
 |-- DAT_INICIO_MATRICULA: string (nullable = true)
 |-- DAT_FIM_MATRICULA: string (nu

In [81]:
df_cd_carteira = (df_cd_carteira.withColumn("DSC_STA_CBRO", f.when(f.col("DSC_MOTIVO").isNull(),"EM COBRANCA")
                                            .otherwise(
                                                f.col("DSC_MOTIVO")
                                            )))                            

In [51]:
df_isencao_total.printSchema()

root
 |-- COD_CONTRATO_ISENCAO: string (nullable = true)
 |-- DSC_MOTIVO_ISENCAO: string (nullable = true)



In [48]:
df_cd_carteira.select(f.col('DSC_STA_CBRO')).show()

+----------------+
|    DSC_STA_CBRO|
+----------------+
|     EM COBRANCA|
|     EM COBRANCA|
|     EM COBRANCA|
|FORA DE COBRANCA|
|     EM COBRANCA|
|     EM COBRANCA|
|     EM COBRANCA|
|     EM COBRANCA|
|     EM COBRANCA|
|     EM COBRANCA|
|     EM COBRANCA|
|     EM COBRANCA|
|     EM COBRANCA|
|     EM COBRANCA|
|     EM COBRANCA|
|     EM COBRANCA|
|     EM COBRANCA|
|     EM COBRANCA|
|     EM COBRANCA|
|     EM COBRANCA|
+----------------+
only showing top 20 rows



In [ ]:
  
    df_cd_carteira = (df_carteira_cobranca.join(df_cd_zeus,
                                  df_carteira_cobranca['COD_CPFCGC_PES'] == df_cd_zeus['COD_RA_ZEUS'], "left"))
    
    

    
    
    #nova coluna - ISENCAO
    df_cd_carteira =  (df_cd_carteira.withColumn("DSC_SITUACAO_COBRANCA", f.when(f.col("CLS_SITUACAO_CONTRATO_ALUNO").isin('12','14','17','20','22','23','33','36','37','Err'),
                                                     f.lit("FORA DE COBRANCA - SITUACAO ACADEMICA"))
                                            .otherwise(
                                                f.col('DSC_MOTIVO_ISENCAO')
                                            ))) 
    
    
    #nova coluna - ISENCAO
    df_cd_carteira =  (df_cd_carteira.withColumn("DSC_SITUACAO_COBRANCA", f.when(f.col("DSC_SITUACAO_COBRANCA").isNull(),
                                                     f.lit("EM COBRANCA"))
                                            .otherwise(
                                                f.col('DSC_SITUACAO_COBRANCA')
                                            )))
                                            
    
                                            
    #NEGATIVADO SERASA 
    df_cd_carteira =  (df_cd_carteira.withColumn("DSC_NEGATIVADO_SERASA", f.when(f.col("NOM_ACAO") == f.lit("Negativacao Serasa"),
                                                     f.lit("SIM"))
                                            .otherwise(
                                                f.lit("NAO")
                                            )))
                                            
    #NEGATIVADO BOA VISTA
    df_cd_carteira =  (df_cd_carteira.withColumn("DSC_NEGATIVADO_BOA_VISTA", f.when(f.col("NOM_ACAO") == f.lit("Negativacao Boa Vista"),
                                                     f.lit("SIM"))
                                            .otherwise(
                                                f.lit("NAO")
                                            )))
                                            
                                            
    df_cd_carteira = df_cd_carteira.dropDuplicates(['COD_CONTRATO'])
    
    df_cd_carteira = df_cd_carteira.dropDuplicates(['COD_CPFCGC_PES'])
    
    return df_cd_carteira